# Differential Attention（差分注意力）

**难度：** Hard

**函数签名：** `diff_attention(Q, K, V, lambda_val) -> Tensor`

**参数：**
- `Q` — 查询张量 (B, S, 2*D_h)
- `K` — 键张量 (B, S, 2*D_h)
- `V` — 值张量 (B, S, D_v)
- `lambda_val` — 标量，控制噪声消除强度

**返回：** 输出张量 (B, S, D_v)

**公式：**
```
out = (softmax(Q1 @ K1.T / √D_h) - λ * softmax(Q2 @ K2.T / √D_h)) @ V
```

**约束：**
- 沿最后一维将 Q/K 拆分为两半
- 使用 `torch.softmax(..., dim=-1)`
- softmax 前除以 √D_h

**提示：** Q1,Q2 = Q[...,:D_h], Q[...,D_h:]；两组注意力图相减消除噪声

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def diff_attention(Q, K, V, lambda_val):
    # ---- Step 1: 拆分 Q 和 K 为两半 ----
    # Q 形状: (B, S, 2*D_h)，沿最后一维切成 Q1 和 Q2
    # Q1 = Q[..., :D_h]  形状: (B, S, D_h) — 第一组查询
    # Q2 = Q[..., D_h:]  形状: (B, S, D_h) — 第二组查询
    # K 同理拆分为 K1, K2
    B, S, D2 = Q.shape
    D_h = D2 // 2
    Q1, Q2 = Q[..., :D_h], Q[..., D_h:]
    K1, K2 = K[..., :D_h], K[..., D_h:]

    # ---- Step 2: 计算缩放因子 ----
    # scale = 1/√D_h，防止点积值过大
    # 这是 Transformer 的标准做法
    scale = D_h ** -0.5

    # ---- Step 3: 计算两组注意力图 ----
    # A1 = softmax(Q1 @ K1^T / √D_h)  形状: (B, S, S)
    # A2 = softmax(Q2 @ K2^T / √D_h)  形状: (B, S, S)
    # 两个独立的注意力分布
    A1 = torch.softmax(Q1 @ K1.transpose(-2, -1) * scale, dim=-1)
    A2 = torch.softmax(Q2 @ K2.transpose(-2, -1) * scale, dim=-1)

    # ---- Step 4: 差分注意力 — 核心创新 ----
    # 标准注意力: A @ V
    # 差分注意力: (A1 - λ * A2) @ V
    # 
    # 为什么相减能消除噪声？
    # - A1 和 A2 都包含「真实注意力 + 噪声」
    # - 相减后，公共的噪声分量被抵消
    # - λ 控制第二组注意力的权重，通常可学习
    # - 当 λ=0 时退化为标准注意力（只用 Q1,K1）
    return (A1 - lambda_val * A2) @ V

In [ ]:
torch.manual_seed(0)
B, S, D2, D_v = 2, 4, 8, 6
Q = torch.randn(B, S, D2)
K = torch.randn(B, S, D2)
V = torch.randn(B, S, D_v)

# lambda=0 should match standard attention on Q1, K1
D_h = D2 // 2
scale = D_h ** -0.5
standard = torch.softmax(Q[..., :D_h] @ K[..., :D_h].transpose(-2, -1) * scale, dim=-1) @ V
diff_zero = diff_attention(Q, K, V, lambda_val=0.0)
print("lambda=0 matches standard attention:", torch.allclose(diff_zero, standard, atol=1e-6))

# lambda=1 with identical halves gives zero output
Q_same = torch.cat([Q[..., :D_h], Q[..., :D_h]], dim=-1)
K_same = torch.cat([K[..., :D_h], K[..., :D_h]], dim=-1)
diff_one = diff_attention(Q_same, K_same, V, lambda_val=1.0)
print("lambda=1 with identical halves gives zero:", torch.allclose(diff_one, torch.zeros_like(diff_one), atol=1e-6))
print("Output shape:", diff_zero.shape)  # (2, 4, 6)

In [ ]:
from torch_judge import check
check("diff_attention")

## 📝 核心思路总结

1. **差分注意力**：两组 softmax 注意力图相减，消除共模噪声
2. **Q/K 拆分**：沿最后一维切成两半，分别计算注意力
3. **lambda 控制**：λ=0 退化为标准注意力，λ=1 完全差分
4. **直觉**：类似差分放大器，两路信号相减消除共模干扰